In [13]:
!pip install qiskit qiskit-algorithms

In [14]:
import sys
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# 2. Add Project Directory to sys.path
project_path = "/content/drive/MyDrive/vanguard-wiser-quantum-portfolio"

# Fallback path check if folder name varies
if not os.path.exists(project_path):
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        if "qubo_builder.py" in files:
            project_path = os.path.dirname(os.path.dirname(os.path.join(root, "qubo_builder.py")))
            break

sys.path.insert(0, project_path)
os.chdir(project_path)
print(f"Active Project Path: {project_path}")

# 3. Imports
import numpy as np
import pandas as pd
from src.qubo_builder import build_portfolio_qubo

# 4. Define Benchmark Inputs
num_assets = 8
target_k = 4
risk_aversion = 0.5
penalty_weight = 100.0

np.random.seed(42)
returns = np.random.uniform(0.05, 0.20, num_assets)
A = np.random.uniform(-0.1, 0.1, (num_assets, num_assets))
cov_matrix = np.dot(A, A.T)

# 5. Build QUBO & Ising Hamiltonian
qaoa_runner, optimal_hamiltonian = build_portfolio_qubo(
    returns=returns,
    cov_matrix=cov_matrix,
    risk_aversion=risk_aversion,
    target_assets=target_k,
    penalty_weight=penalty_weight
)

# 6. Classical Ground State (Exact Matrix Diagonalization)
hamiltonian_matrix = optimal_hamiltonian.to_matrix()
exact_eigenvalues = np.linalg.eigvalsh(hamiltonian_matrix)
exact_energy = float(np.min(exact_eigenvalues).real)

# 7. QAOA Ground State Execution
qaoa_result = qaoa_runner.compute_minimum_eigenvalue(operator=optimal_hamiltonian)
qaoa_energy = float(qaoa_result.eigenvalue.real)

# 8. Extract Selection & Check Feasibility
best_measurement = getattr(qaoa_result, 'best_measurement', None)
if best_measurement and 'state' in best_measurement:
    state_val = best_measurement['state']
    bitstring = state_val if isinstance(state_val, str) else format(int(state_val), f'0{num_assets}b')
else:
    best_idx = np.argmax(np.abs(qaoa_result.eigenstate.data)**2)
    bitstring = format(best_idx, f'0{num_assets}b')

binary_selection = [int(bitstring[num_assets - 1 - i]) for i in range(num_assets)]
assets_selected_count = sum(binary_selection)

# 9. Benchmark Summary Output
approx_ratio = (qaoa_energy / exact_energy) if exact_energy != 0 else 1.0

benchmark_df = pd.DataFrame([{
    "Solver": "Classical Exact (NumPy Diagonalization)",
    "Ground Energy": f"{exact_energy:.4f}",
    "Assets Selected": f"{target_k}/{target_k}",
    "Feasible": True,
    "Approximation Ratio": "100.0%"
}, {
    "Solver": "QAOA (p=3, StatevectorSampler)",
    "Ground Energy": f"{qaoa_energy:.4f}",
    "Assets Selected": f"{assets_selected_count}/{target_k}",
    "Feasible": assets_selected_count == target_k,
    "Approximation Ratio": f"{approx_ratio * 100:.2f}%"
}])

print("--- Quantum vs Classical Benchmark ---")
display(benchmark_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Active Project Path: /content/drive/MyDrive/vanguard-wiser-quantum-portfolio


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


--- Quantum vs Classical Benchmark ---


,Solver,Ground Energy,Assets Selected,Feasible,Approximation Ratio
0,Classical Exact (NumPy Diagonalization),-200.1089,4/4,True,100.0%
1,"QAOA (p=3, StatevectorSampler)",-24.5960,4/4,True,12.29%


In [ ]:
# 1. Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 2. Start Streamlit in background
!streamlit run /content/drive/MyDrive/vanguard-wiser-quantum-portfolio/app/portfolio_copilot.py --server.port 8501 --server.address 0.0.0.0 > /dev/null 2>&1 &

# 3. Launch Cloudflare Tunnel (Copy the trycloudflare.com URL generated below)
!./cloudflared tunnel --url http://localhost:8501

2026-08-05T17:30:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-05T17:30:03Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-05T17:30:11Z INF +--------------------------------------------------------------------------------------------+
2026-08-05T17:30:11Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-05T17:30:11Z INF |  https://mapping-rug-pocket-sierra.trycloudflare.com  